In [1]:
%autosave 60
%pip install --quiet -r requirements.txt

Autosaving every 60 seconds
Note: you may need to restart the kernel to use updated packages.


In [2]:
import yaml
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

from utils.data_loader import DataLoader
from utils.health_data_loader import HealthDataLoader
from utils.data_converter import DataConverter
from utils.data_analyser import DataAnalyser
from models.unet_3d_std import UNet3D
from models.unet_3d_film import FiLMUNet3D
from utils.grid_search import create_hyperparameter_grid, run_pipeline, select_champions
from utils.train_eval import fit_3D, fit_feature_based_3D, get_middle_slice_3D
from utils.loss_functions import ssim_loss, l1_loss

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda:0


## 1. Data Setup

In [3]:
root_path     = 'data/'
metadata_root = 'data/metadata'
grid_path     = 'data/grid_search.yaml'

data_loader    = DataLoader(root_path=root_path)
meta_loader    = HealthDataLoader(root_path=metadata_root, external_features=False)
data_converter = DataConverter()
data_analyser  = DataAnalyser(root_path=root_path)

# Build / refresh metadata_combined.csv
metadata = meta_loader.combine_metadata()
metadata.head()

[combine_metadata] 736 rows, 37 cols → data/metadata/metadata_combined.csv
  94 row(s) missing fastsurfer — filled with 0
  33 row(s) missing mock_metadata — filled with 0


,hunt_id,wmh_volume,frontal_thickness_mean,frontal_volume_total,parietal_thickness_mean,parietal_volume_total,temporal_thickness_mean,temporal_volume_total,occipital_thickness_mean,occipital_volume_total,...,self_rated_health_excellent,self_rated_health_fair,self_rated_health_good,self_rated_health_poor,self_rated_health_very_good,smoking_status_current,smoking_status_former,smoking_status_never,pack_years,had_score
0,00039,0.022649,0.581530,0.591519,0.571508,0.379907,0.766420,0.603967,0.260179,0.630277,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.221757,0.449814
1,00046,0.035594,0.535797,0.782791,0.518232,0.517416,0.723274,0.691759,0.222411,0.705390,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.000000,0.182156
2,00053,0.152173,0.571785,0.809829,0.546830,0.417651,0.741640,0.720567,0.186556,0.654447,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.357741,0.070632
3,00077,0.039974,0.535852,0.627646,0.570404,0.402320,0.796430,0.615629,0.251702,0.697597,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.080544,0.375465
4,00091,0.054953,0.582312,0.660934,0.404700,0.349206,0.733089,0.616569,0.231783,0.603187,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.000000,0.189591


### Split dataset based on features

In [4]:
from utils.combined_metadata_loader import CombinedMetadataLoader

available_ids = set(data_loader.all_candidates)
split_loader  = HealthDataLoader(root_path=metadata_root, external_features=True)
train_ids, val_ids, test_ids = split_loader.split(train_split=0.70, val_split=0.15, seed=69)

training_pairs   = [data_loader.get_pair_path_from_id(i) for i in train_ids  if i in available_ids]
validation_pairs = [data_loader.get_pair_path_from_id(i) for i in val_ids    if i in available_ids]
test_pairs       = [data_loader.get_pair_path_from_id(i) for i in test_ids   if i in available_ids]

cond_loader = CombinedMetadataLoader(csv_path=f'{metadata_root}/metadata_combined.csv')
COND_DIM    = cond_loader.n_features

print(f'Train: {len(training_pairs)}  Val: {len(validation_pairs)}  Test: {len(test_pairs)}')
print(f'Conditioning vector size: {COND_DIM}')

Train: 482  Val: 108  Test: 113
Conditioning vector size: 36


In [5]:
create_hyperparameter_grid(grid_file=grid_path, epochs=1) 

[create_hyperparameter_grid] data/grid_search.yaml already exists — skipping.


## 2. Run Models

In [ ]:
run_pipeline(grid_file=grid_path, device=device, metadata_root=metadata_root, train_pairs=training_pairs, val_pairs=validation_pairs)

[pipeline] Skipping std_001 — already completed
[pipeline] Skipping std_002 — already completed
[pipeline] Skipping std_003 — already completed
[pipeline] Skipping std_004 — already completed
[pipeline] Skipping std_005 — already completed
[pipeline] Skipping std_006 — already completed
[pipeline] Skipping std_007 — already completed
[pipeline] Skipping std_008 — already completed
[pipeline] Skipping std_009 — already completed
[pipeline] Skipping std_010 — already completed
[pipeline] Skipping std_011 — already completed
[pipeline] Skipping std_012 — already completed
[pipeline] Skipping std_013 — already completed
[pipeline] Skipping std_014 — already completed
[pipeline] Skipping std_015 — already completed
[pipeline] Skipping std_016 — already completed
[pipeline] Skipping film_simple_001 — already completed
[pipeline] Skipping film_simple_002 — already completed
[pipeline] Skipping film_simple_003 — already completed
[pipeline] Skipping film_simple_004 — already completed
[pipelin

Training 3D Residual U-Net:   0%|          | 0/1 [00:00<?, ?it/s]

## 4. Analysis

Run these cells after all three models are trained (or loaded from disk).

In [ ]:
with open(grid_path) as f:
    grid_doc = yaml.safe_load(f)

completed = [e for e in grid_doc['grid'] if e.get('results', {}).get('completed')]
print(f"Completed: {len(completed)} / {len(grid_doc['grid'])} entries\n")

rows = []
for e in completed:
    r = e['results']
    rows.append({
        'id':            e['id'],
        'model_type':    e['model_type'],
        'film_type':     e.get('film_generator_type', '—'),
        'lr':            e['learning_rate'],
        'weight_decay':  e['weight_decay'],
        'scheduler':     e['lr_scheduler'],
        'mlp_hidden':    e.get('mlp_hidden', '—'),
        'best_val_loss': r['best_val_loss'],
        'trained_at':    r.get('trained_at', ''),
    })

df = pd.DataFrame(rows).sort_values('best_val_loss').reset_index(drop=True)
display(df)

Completed: 80 / 80 entries



,id,model_type,film_type,lr,weight_decay,scheduler,mlp_hidden,best_val_loss,trained_at
0,std_003,std,—,0.00005,0.00001,cosine,—,0.784144,2026-04-20T13:44:47
1,std_004,std,—,0.00005,0.00001,plateau,—,0.821208,2026-04-20T13:46:32
2,std_001,std,—,0.00005,0.00001,cosine,—,0.828915,2026-04-20T13:33:21
3,film_complex_032,film,complex,0.00010,0.00010,plateau,128,0.836862,2026-04-20T13:59:57
4,std_005,std,—,0.00005,0.00010,cosine,—,0.838555,2026-04-20T13:48:17
...,...,...,...,...,...,...,...,...,...
75,film_complex_026,film,complex,0.00010,0.00010,cosine,128,0.957921,2026-04-20T13:50:06
76,film_complex_028,film,complex,0.00010,0.00010,plateau,128,0.957921,2026-04-20T13:50:06
77,film_complex_029,film,complex,0.00010,0.00010,cosine,64,0.957921,2026-04-20T13:50:06
78,film_complex_030,film,complex,0.00010,0.00010,cosine,128,0.957921,2026-04-20T13:50:06


In [ ]:
champions = select_champions(grid_file=grid_path, device=device, metadata_root=metadata_root)
print(champions)

[select_champions] std: std_003  val_loss=0.784144  ← out/grid_search/std_003_best.pth


RuntimeError: Error(s) in loading state_dict for FiLMUNet3D:
	Missing key(s) in state_dict: "e1.conv1.weight", "e1.conv2.weight", "e2.conv.conv1.weight", "e2.conv.conv2.weight", "e3.conv.conv1.weight", "e3.conv.conv2.weight", "e4.conv.conv1.weight", "e4.conv.conv2.weight", "b1.conv1.weight", "b1.conv2.weight", "b2.conv1.weight", "b2.conv2.weight", "u4.conv.conv1.weight", "u4.conv.conv2.weight", "u3.conv.conv1.weight", "u3.conv.conv2.weight", "u2.conv.conv1.weight", "u2.conv.conv2.weight", "u1.conv.conv1.weight", "u1.conv.conv2.weight", "film_gen.mlp.0.weight", "film_gen.mlp.0.bias", "film_gen.mlp.2.weight", "film_gen.mlp.2.bias", "film_gen.mlp.4.weight", "film_gen.mlp.4.bias". 
	Unexpected key(s) in state_dict: "bott.block.0.weight", "bott.block.3.weight", "bott_sec.block.0.weight", "bott_sec.block.3.weight", "e1.block.0.weight", "e1.block.3.weight", "e2.conv.block.0.weight", "e2.conv.block.3.weight", "e3.conv.block.0.weight", "e3.conv.block.3.weight", "e4.conv.block.0.weight", "e4.con

### Test Set Evaluation

In [ ]:
def evaluate_model(model, pairs, conditional=False):
    model.eval()
    ssim_vals, l1_vals = [], []

    with torch.no_grad():
        for x_path, y_path in tqdm(pairs, desc='Evaluating'):
            x_full = data_converter.load_path_as_tensor(x_path, device)
            y_full = data_converter.load_path_as_tensor(y_path, device)
            x = data_converter.get_volume_with_3d_change(x_full, ((16, 10, 0), (17, 11, 17)), remove_mode=True)
            y = data_converter.get_volume_with_3d_change(y_full, ((16, 10, 0), (17, 11, 17)), remove_mode=True)

            if conditional:
                cond = torch.tensor(cond_loader.get(x_path).tolist(), dtype=x.dtype).unsqueeze(0).to(device)
                out  = model(x, cond)
            else:
                out = model(x)

            y_hat = out[0] if isinstance(out, (tuple, list)) else out
            ssim_vals.append(ssim_loss(y_hat, y).item())
            l1_vals.append(l1_loss(y_hat, y).item())
            del x_full, y_full, x, y, out, y_hat

    return dict(
        ssim_mean=float(np.mean(ssim_vals)),
        ssim_std=float(np.std(ssim_vals)),
        l1_mean=float(np.mean(l1_vals)),
        l1_std=float(np.std(l1_vals)),
    )


print('Evaluating Standard U-Net...')
std_test = evaluate_model(std_best, test_pairs, conditional=False)

print('Evaluating FiLM Simple...')
film_simple_test = evaluate_model(film_simple_best, test_pairs, conditional=True)

print('Evaluating FiLM Complex...')
film_complex_test = evaluate_model(film_complex_best, test_pairs, conditional=True)

In [ ]:
test_results = {
    'Standard U-Net': std_test,
    'FiLM Simple':    film_simple_test,
    'FiLM Complex':   film_complex_test,
}

header = '{:<20}  {:>12}  {:>10}  {:>12}  {:>10}'.format(
    'Model', 'SSIM Loss', 'SSIM Std', 'L1 Loss', 'L1 Std')
print(header)
print('-' * len(header))
for name, r in test_results.items():
    print('{:<20}  {:>12.5f}  {:>10.5f}  {:>12.5f}  {:>10.5f}'.format(
        name, r['ssim_mean'], r['ssim_std'], r['l1_mean'], r['l1_std']))

In [ ]:
names  = list(test_results.keys())
colors = ['#4C72B0', '#DD8452', '#55A868']

ssim_means = [test_results[n]['ssim_mean'] for n in names]
ssim_stds  = [test_results[n]['ssim_std']  for n in names]
l1_means   = [test_results[n]['l1_mean']   for n in names]
l1_stds    = [test_results[n]['l1_std']    for n in names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.bar(names, ssim_means, yerr=ssim_stds, color=colors, capsize=6, width=0.5, edgecolor='white')
ax1.set_title('Test SSIM Loss  (lower = better)', fontsize=12)
ax1.set_ylabel('1 - SSIM')
ax1.set_ylim(bottom=0)
ax1.grid(axis='y', alpha=0.3)

ax2.bar(names, l1_means, yerr=l1_stds, color=colors, capsize=6, width=0.5, edgecolor='white')
ax2.set_title('Test L1 Loss  (lower = better)', fontsize=12)
ax2.set_ylabel('Mean Absolute Error')
ax2.set_ylim(bottom=0)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### Visual Reconstruction Comparison

In [ ]:
x_path, y_path = test_pairs[0]

with torch.no_grad():
    x_full = data_converter.load_path_as_tensor(x_path, device)
    y_full = data_converter.load_path_as_tensor(y_path, device)
    x = data_converter.get_volume_with_3d_change(x_full, ((16, 10, 0), (17, 11, 17)), remove_mode=True)
    y = data_converter.get_volume_with_3d_change(y_full, ((16, 10, 0), (17, 11, 17)), remove_mode=True)

    cond = torch.tensor(cond_loader.get(x_path).tolist(), dtype=x.dtype).unsqueeze(0).to(device)

    def _recon(model, conditional):
        out = model(x, cond) if conditional else model(x)
        return out[0] if isinstance(out, (tuple, list)) else out

    std_hat = _recon(std_best,          conditional=False)
    fs_hat  = _recon(film_simple_best,  conditional=True)
    fc_hat  = _recon(film_complex_best, conditional=True)

x_sl   = get_middle_slice_3D(x)
y_sl   = get_middle_slice_3D(y)
std_sl = get_middle_slice_3D(std_hat)
fs_sl  = get_middle_slice_3D(fs_hat)
fc_sl  = get_middle_slice_3D(fc_hat)

panels = [
    ('HUNT3 (input)',  x_sl,   None),
    ('HUNT4 (target)', y_sl,   None),
    ('Standard U-Net', std_sl, std_sl),
    ('FiLM Simple',    fs_sl,  fs_sl),
    ('FiLM Complex',   fc_sl,  fc_sl),
]

fig, axes = plt.subplots(2, 5, figsize=(22, 8))

for col, (title, sl, recon_sl) in enumerate(panels):
    axes[0, col].imshow(sl, cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(title, fontsize=11, fontweight='bold')
    axes[0, col].axis('off')

    if recon_sl is not None:
        diff = np.abs(recon_sl - y_sl)
        axes[1, col].imshow(diff, cmap='hot', vmin=0, vmax=0.3)
        axes[1, col].set_title('Diff  MAE={:.4f}'.format(np.mean(diff)), fontsize=10)
        axes[1, col].axis('off')
    else:
        axes[1, col].set_visible(False)

fig.suptitle('Visual Comparison — First Test Subject', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()